# Project Additional Materials — Daily AutoML Models (XGB / RF / LGB)

- Student ID: 10841269  
- Course Code: DATA70132  
- Academic Year: 2024–25  

**Environment:** See `README` and `ERP_Environment_2025.yaml`.  
**Reproduction:** Run this notebook top to bottom in the same folder as `All_UKonly_daily_cleaned_ymd.nc`.  

This notebook trains AutoML models (XGBoost, Random Forest, LightGBM via FLAML) on daily data and saves the fitted models and logs.

## Step 0 — Import required libraries
Load the Python packages required for daily AutoML model training.  
(Dependencies are listed in the README and `ERP_Environment_2025.yaml`.)


In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import random
import warnings
import pickle
from flaml import AutoML
from sklearn.metrics import r2_score, mean_squared_error
warnings.filterwarnings('ignore')

## Step 1 — Load daily dataset & prepare training data
Load `All_UKonly_daily_cleaned_ymd.nc` and define features and target.  
Set time-based training window: 2006–2020.


In [ ]:
# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

# Load dataset
ds = xr.open_dataset("All_UKonly_daily_cleaned_ymd.nc", decode_times=False)
df = ds.to_dataframe().dropna().reset_index()

# Time split
df_train = df[(df["year"] >= 2006) & (df["year"] <= 2020)]

# Define features and target
features = ["TREFHT", "FLNS", "FSNS", "QBOT", "UBOT", "VBOT", "PRECT", "PRSN"]
target = "TREFMXAV_U"

# Prepare training data
X_train, y_train = df_train[features], df_train[target]

## Step 2 — Train XGBoost AutoML model with FLAML
Train the models and select the best one to save `xgb_daily.pkl`.


In [ ]:
# XGBoost AutoML model
automl_xgb = AutoML()
automl_xgb.fit(
    X_train, y_train,
    task="regression", # Regression task
    estimator_list=["xgboost"], # Use only XGBoost
    time_budget=3600, # 1 hour
    metric="r2", # Optimize for R^2
    eval_method="cv", # Cross-validation
    n_jobs=-1, # Use all available cores
    log_file_name="xgb_daily.log" # Log file name
)

print("\n XGBoost Best Config:", automl_xgb.best_config)
print("Validation Loss:", automl_xgb.best_loss)

# Save model
model_save_path = 'xgb_daily.pkl'
with open(f"{model_save_path}", "wb") as f:
    pickle.dump(automl_xgb, f, pickle.HIGHEST_PROTOCOL)
print(f"Model for UK urban temperature prediction saved as {model_save_path}")

[flaml.automl.logger: 09-22 00:54:51] {1752} INFO - task = regression
[flaml.automl.logger: 09-22 00:54:51] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 09-22 00:54:53] {1862} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 09-22 00:54:53] {1979} INFO - List of ML learners in AutoML Run: ['xgboost']
[flaml.automl.logger: 09-22 00:54:53] {2282} INFO - iteration 0, current learner xgboost
[flaml.automl.logger: 09-22 00:55:09] {2417} INFO - Estimated sufficient time budget=157227s. Estimated necessary time budget=157s.
[flaml.automl.logger: 09-22 00:55:09] {2466} INFO -  at 35.5s,	estimator xgboost's best error=0.5292,	best estimator xgboost's best error=0.5292
[flaml.automl.logger: 09-22 00:55:09] {2282} INFO - iteration 1, current learner xgboost
[flaml.automl.logger: 09-22 00:55:25] {2466} INFO -  at 52.2s,	estimator xgboost's best error=0.5292,	best estimator xgboost's best error=0.5292
[flaml.automl.logger: 09-22 00:55:25] {2282} INFO - iteration 2, current le

## Step 3 — Train Random Forest AutoML model with FLAML
Train the models and select the best one to save `rf_daily.pkl`.


In [ ]:
# RF AutoML model
automl_rf = AutoML()
automl_rf.fit(
    X_train, y_train, # training data
    task="regression", # Regression task
    estimator_list=["rf"], # Use only Random Forest
    time_budget=3600, # 1 hour
    metric="r2", # Optimize for R^2
    eval_method="cv", # Cross-validation
    n_jobs=-1, # Use all available cores
    log_file_name="rf_daily.log" # Log file name
)

print("\n Random Forest Best Config:", automl_rf.best_config)
print("Validation Loss:", automl_rf.best_loss)

# Save model
model_save_path = 'rf_daily.pkl'
with open(f"{model_save_path}", "wb") as f:
    pickle.dump(automl_rf, f, pickle.HIGHEST_PROTOCOL)
print(f"Model for UK urban temperature prediction saved as {model_save_path}")

[flaml.automl.logger: 09-22 07:55:55] {1752} INFO - task = regression
[flaml.automl.logger: 09-22 07:55:55] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 09-22 07:55:56] {1862} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 09-22 07:55:56] {1979} INFO - List of ML learners in AutoML Run: ['rf']
[flaml.automl.logger: 09-22 07:55:56] {2282} INFO - iteration 0, current learner rf
[flaml.automl.logger: 09-22 07:58:24] {2417} INFO - Estimated sufficient time budget=1477530s. Estimated necessary time budget=1478s.
[flaml.automl.logger: 09-22 07:58:24] {2466} INFO -  at 166.5s,	estimator rf's best error=0.1849,	best estimator rf's best error=0.1849
[flaml.automl.logger: 09-22 07:58:24] {2282} INFO - iteration 1, current learner rf
[flaml.automl.logger: 09-22 08:02:10] {2466} INFO -  at 392.3s,	estimator rf's best error=0.0891,	best estimator rf's best error=0.0891
[flaml.automl.logger: 09-22 08:02:10] {2282} INFO - iteration 2, current learner rf
[flaml.automl.logger: 

## Step 4 — Train LightGBM AutoML model with FLAML
Train the models and select the best one to `lgb_daily.pkl`.


In [ ]:
# LightGBM AutoML model
automl_lgb = AutoML()
automl_lgb.fit(
    X_train, y_train, # training data
    task="regression", # Regression task
    estimator_list=["lgbm"], # Use only LightGBM
    time_budget=3600, # 1 hour
    metric="r2", # Optimize for R^2
    eval_method="cv", # Cross-validation
    n_jobs=-1, # Use all available cores
    log_file_name="lgb_daily.log" # Log file name
)

print("\n LightGBM Best Config:", automl_lgb.best_config)
print("Validation Loss:", automl_lgb.best_loss)

# Save model
model_save_path = 'lgb_daily.pkl'
with open(f"{model_save_path}", "wb") as f:
    pickle.dump(automl_lgb, f, pickle.HIGHEST_PROTOCOL)
print(f"Model for UK urban temperature prediction saved as {model_save_path}")

[flaml.automl.logger: 09-06 19:39:26] {1752} INFO - task = regression
[flaml.automl.logger: 09-06 19:39:26] {1763} INFO - Evaluation method: cv
[flaml.automl.logger: 09-06 19:39:27] {1862} INFO - Minimizing error metric: 1-r2
[flaml.automl.logger: 09-06 19:39:27] {1979} INFO - List of ML learners in AutoML Run: ['lgbm']
[flaml.automl.logger: 09-06 19:39:28] {2282} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 09-06 19:39:40] {2417} INFO - Estimated sufficient time budget=121935s. Estimated necessary time budget=122s.
[flaml.automl.logger: 09-06 19:39:40] {2466} INFO -  at 30.3s,	estimator lgbm's best error=0.5294,	best estimator lgbm's best error=0.5294
[flaml.automl.logger: 09-06 19:39:40] {2282} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 09-06 19:39:52] {2466} INFO -  at 42.9s,	estimator lgbm's best error=0.5294,	best estimator lgbm's best error=0.5294
[flaml.automl.logger: 09-06 19:39:52] {2282} INFO - iteration 2, current learner lgbm
[flaml.aut